# Student Attendance Modeling (Regression + Classification)
This notebook uses a clean 1000-record dataset and trains multiple models for both tasks:
- Regression: predict `Attendance_Percentage`
- Classification: predict `Attendance_Label` (High/Low)


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    accuracy_score, f1_score, classification_report, confusion_matrix
)

from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.svm import SVR, SVC
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier, GradientBoostingClassifier

sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)


In [ ]:
df = pd.read_csv('attendance.csv')
df.head()


In [ ]:
print('Shape:', df.shape)
df.info()


In [ ]:
df.describe().T


In [ ]:
# Missing values check
df.isnull().sum()


In [ ]:
# Target distribution plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['Attendance_Percentage'], bins=30, kde=True, ax=axes[0], color='teal')
axes[0].set_title('Attendance Percentage Distribution')

sns.countplot(data=df, x='Attendance_Label', ax=axes[1], palette='Set2')
axes[1].set_title('Attendance Label Distribution')

plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap
numeric_df = df.drop(columns=['Attendance_Label'])
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()


In [ ]:
feature_cols = [
    'Math_Score','Science_Score','English_Score','Study_Hours_Per_Week',
    'Assignments_Completed','Participation_Score','Absences','Previous_Attendance_Percentage'
]

X = df[feature_cols]
y_reg = df['Attendance_Percentage']
y_clf = df['Attendance_Label']


In [ ]:
# Train/test split for regression
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

reg_models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.001),
    'KNNRegressor': KNeighborsRegressor(n_neighbors=7),
    'SVR': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVR(C=10, epsilon=0.1, kernel='rbf'))
    ]),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=250, random_state=42),
    'GradientBoostingRegressor': GradientBoostingRegressor(random_state=42)
}

reg_results = []
reg_predictions = {}

for name, model in reg_models.items():
    model.fit(X_train_r, y_train_r)
    pred = model.predict(X_test_r)
    reg_predictions[name] = pred
    reg_results.append({
        'Model': name,
        'R2': r2_score(y_test_r, pred),
        'MAE': mean_absolute_error(y_test_r, pred),
        'RMSE': mean_squared_error(y_test_r, pred, squared=False)
    })

reg_results_df = pd.DataFrame(reg_results).sort_values(by='R2', ascending=False)
reg_results_df


In [ ]:
# Regression model comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(data=reg_results_df, x='R2', y='Model', ax=axes[0], palette='viridis')
axes[0].set_title('Regression Models by R2 (Higher is Better)')

sorted_rmse = reg_results_df.sort_values(by='RMSE', ascending=True)
sns.barplot(data=sorted_rmse, x='RMSE', y='Model', ax=axes[1], palette='magma')
axes[1].set_title('Regression Models by RMSE (Lower is Better)')

plt.tight_layout()
plt.show()


In [ ]:
# Actual vs predicted plot for best regression model
best_reg = reg_results_df.iloc[0]['Model']
plt.figure(figsize=(8, 6))
plt.scatter(y_test_r, reg_predictions[best_reg], alpha=0.7)
plt.plot([y_test_r.min(), y_test_r.max()], [y_test_r.min(), y_test_r.max()], 'r--')
plt.xlabel('Actual Attendance')
plt.ylabel('Predicted Attendance')
plt.title(f'Actual vs Predicted ({best_reg})')
plt.tight_layout()
plt.show()

print('Best regression model:', best_reg)


In [ ]:
# Train/test split for classification
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

clf_models = {
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=2000, random_state=42))
    ]),
    'KNNClassifier': KNeighborsClassifier(n_neighbors=7),
    'SVC': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', C=3, gamma='scale', random_state=42))
    ]),
    'RandomForestClassifier': RandomForestClassifier(n_estimators=250, random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42)
}

clf_results = []
clf_predictions = {}

for name, model in clf_models.items():
    model.fit(X_train_c, y_train_c)
    pred = model.predict(X_test_c)
    clf_predictions[name] = pred
    clf_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test_c, pred),
        'F1_Macro': f1_score(y_test_c, pred, average='macro')
    })

clf_results_df = pd.DataFrame(clf_results).sort_values(by='Accuracy', ascending=False)
clf_results_df


In [ ]:
# Classification model comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(data=clf_results_df, x='Accuracy', y='Model', ax=axes[0], palette='Blues_r')
axes[0].set_title('Classification Models by Accuracy')
axes[0].set_xlim(0, 1)

sorted_f1 = clf_results_df.sort_values(by='F1_Macro', ascending=False)
sns.barplot(data=sorted_f1, x='F1_Macro', y='Model', ax=axes[1], palette='Greens_r')
axes[1].set_title('Classification Models by F1 Macro')
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.show()


In [ ]:
# Detailed report for best classification model
best_clf = clf_results_df.iloc[0]['Model']
best_pred = clf_predictions[best_clf]

print('Best classification model:', best_clf)
print('Classification Report:')
print(classification_report(y_test_c, best_pred))

cm = confusion_matrix(y_test_c, best_pred, labels=['Low', 'High'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu', xticklabels=['Low', 'High'], yticklabels=['Low', 'High'])
plt.title(f'Confusion Matrix ({best_clf})')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()


In [ ]:
# Example prediction for a new student
sample_student = pd.DataFrame([{
    'Math_Score': 82,
    'Science_Score': 78,
    'English_Score': 85,
    'Study_Hours_Per_Week': 14,
    'Assignments_Completed': 17,
    'Participation_Score': 8,
    'Absences': 4,
    'Previous_Attendance_Percentage': 84.5
}])

# Re-fit best models on full training splits for a direct example
best_reg_model = reg_models[best_reg]
best_reg_model.fit(X_train_r, y_train_r)
reg_pred_value = best_reg_model.predict(sample_student)[0]

best_clf_model = clf_models[best_clf]
best_clf_model.fit(X_train_c, y_train_c)
clf_pred_value = best_clf_model.predict(sample_student)[0]

print('Predicted Attendance_Percentage:', round(float(reg_pred_value), 2))
print('Predicted Attendance_Label:', clf_pred_value)
